<a href="https://colab.research.google.com/github/Maham-Anees/Python/blob/main/Air_Quality_Data_India_2015_2020.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# The file path was provided in the context.
file_path = '/content/city_day.csv.zip'

# Read the zipped CSV file directly into a pandas DataFrame.
df = pd.read_csv(file_path)

# Display the first 5 rows to ensure it loaded correctly.
display(df.head())

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/city_day.csv.zip') # Changed file path to the correct zipped file
df.shape
df.head()
df.info()

In [ ]:
print(df.isnull().sum())

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])

df['Month'] = df['Date'].dt.month

df['Year'] = df['Date'].dt.year

df['AQI'] = df['AQI'].fillna(df['AQI'].median())

In [ ]:
df.describe()
df['AQI'].describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for professional look
sns.set_style("whitegrid")
sns.set_palette("muted")

plt.figure(figsize=(10, 6))

# Histogram with KDE
sns.histplot(df['AQI'], bins=50, kde=True, color='steelblue', edgecolor='white', linewidth=0.5)

# Add vertical lines for AQI category boundaries
categories = [50, 100, 200, 300, 400]
colors     = ['green', 'yellowgreen', 'orange', 'red', 'darkred']
labels     = ['Good', 'Satisfactory', 'Moderate', 'Poor', 'Very Poor']

for val, col, lbl in zip(categories, colors, labels):
    plt.axvline(x=val, color=col, linestyle='--', linewidth=1.2, label=f'{lbl} ({val})')

plt.title('Distribution of Air Quality Index (AQI) — All Indian Cities (2015–2020)',
          fontsize=14, fontweight='bold', pad=15)
plt.xlabel('AQI Value', fontsize=12)
plt.ylabel('Number of Daily Readings', fontsize=12)
plt.legend(title='AQI Categories', fontsize=9, title_fontsize=10)
plt.grid(axis='y', alpha=0.4)
plt.tight_layout()

# Save for your report
plt.savefig('Figure_4_1_AQI_Distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved!")

In [ ]:
from scipy import stats

# Filter data for Delhi and Mumbai
aqi_delhi = df[df['City'] == 'Delhi']['AQI'].dropna()
aqi_mumbai = df[df['City'] == 'Mumbai']['AQI'].dropna()

# Perform independent t-test
t_stat, p_value = stats.ttest_ind(aqi_delhi, aqi_mumbai, equal_var=False) # Welch's t-test assuming unequal variances

print(f"Mean AQI for Delhi: {aqi_delhi.mean():.2f}")
print(f"Mean AQI for Mumbai: {aqi_mumbai.mean():.2f}")
print(f"T-statistic: {t_stat:.2f}")
print(f"P-value: {p_value:.3f}")

# Interpret the p-value
alpha = 0.05 # Significance level
if p_value < alpha:
    print("\nConclusion: Reject the null hypothesis. There is a statistically significant difference in mean AQI between Delhi and Mumbai.")
else:
    print("\nConclusion: Fail to reject the null hypothesis. There is no statistically significant difference in mean AQI between Delhi and Mumbai.")

### Interpretation:
*   **Null Hypothesis (H0):** There is no significant difference in the mean AQI between Delhi and Mumbai.
*   **Alternative Hypothesis (Ha):** There is a significant difference in the mean AQI between Delhi and Mumbai.

We typically use a significance level (alpha) of 0.05. If the p-value is less than alpha, we reject the null hypothesis, suggesting a significant difference. Otherwise, we fail to reject the null hypothesis.

In [ ]:
print(df['City'].unique())

After running the above cell, you will be prompted to authorize access to your Google Drive. Follow the instructions to complete the mounting process. Once mounted, your Google Drive contents will be accessible at `/content/drive`.

In [ ]:
# Add Season column
# Ensure 'Date' is datetime and extract 'Month' and 'Year' if not already done
import pandas as pd

if 'Date' in df.columns and not pd.api.types.is_datetime64_any_dtype(df['Date']):
    df['Date'] = pd.to_datetime(df['Date'])

if 'Month' not in df.columns and 'Date' in df.columns:
    df['Month'] = df['Date'].dt.month

if 'Year' not in df.columns and 'Date' in df.columns:
    df['Year'] = df['Date'].dt.year

def get_season(month):
    if month in [10, 11, 12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Summer'
    else:
        return 'Monsoon'

df['Season'] = df['Month'].apply(get_season)

# Figure 4.2 — City-wise Average AQI
city_avg = df.groupby('City')['AQI'].mean().sort_values(ascending=True)

plt.figure(figsize=(12, 8))
bars = plt.barh(city_avg.index, city_avg.values, color='steelblue', edgecolor='white')

# Color top 5 red, bottom 5 green
for i, bar in enumerate(bars):
    if i >= len(bars) - 5:
        bar.set_color('#e05c5c')
    elif i < 5:
        bar.set_color('#5cad6e')

plt.axvline(x=100, color='orange', linestyle='--', linewidth=1, label='Satisfactory limit (100)')
plt.axvline(x=200, color='red', linestyle='--', linewidth=1, label='Poor threshold (200)')
plt.title('Average AQI by City — India (2015–2020)', fontsize=14, fontweight='bold')
plt.xlabel('Average AQI', fontsize=12)
plt.legend(fontsize=9)
plt.tight_layout()
plt.savefig('Figure_4_2_City_AQI.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved Figure 4.2")

In [ ]:
# Ensure 'Date' is datetime and extract 'Month' and 'Year' if not already done
import pandas as pd

if 'Date' in df.columns and not pd.api.types.is_datetime64_any_dtype(df['Date']):
    df['Date'] = pd.to_datetime(df['Date'])

if 'Month' not in df.columns and 'Date' in df.columns:
    df['Month'] = df['Date'].dt.month

if 'Year' not in df.columns and 'Date' in df.columns:
    df['Year'] = df['Date'].dt.year

# Fill missing AQI values with the median, as done in previous preprocessing
df['AQI'] = df['AQI'].fillna(df['AQI'].median())

# Define city_avg
city_avg = df.groupby('City')['AQI'].mean().sort_values(ascending=True)

# Figure 4.4 — Year-wise AQI Trend (Top 5 Polluted Cities)
top5 = city_avg.sort_values(ascending=False).head(5).index.tolist()
yearly = df[df['City'].isin(top5)].groupby(['Year','City'])['AQI'].mean().reset_index()

plt.figure(figsize=(11, 6))
for city in top5:
    data = yearly[yearly['City'] == city]
    plt.plot(data['Year'], data['AQI'], marker='o', linewidth=2, label=city)

plt.title('Year-wise AQI Trend — Top 5 Most Polluted Cities', fontsize=14, fontweight='bold')
plt.xlabel('Year', fontsize=12)
plt.ylabel('Average AQI', fontsize=12)
plt.legend(title='City', fontsize=10)
plt.grid(alpha=0.4)
plt.tight_layout()
plt.savefig('Figure_4_4_Yearly_Trend.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved Figure 4.4")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Ensure 'Date' is datetime and extract 'Month' and 'Year' if not already done
if 'Date' in df.columns and not pd.api.types.is_datetime64_any_dtype(df['Date']):
    df['Date'] = pd.to_datetime(df['Date'])

if 'Month' not in df.columns and 'Date' in df.columns:
    df['Month'] = df['Date'].dt.month

if 'Year' not in df.columns and 'Date' in df.columns:
    df['Year'] = df['Date'].dt.year

# Figure 4.3 — Monthly AQI Trend for Delhi
delhi = df[df['City'] == 'Delhi'].copy()
monthly_delhi = delhi.groupby('Month')['AQI'].mean()

month_names = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']

plt.figure(figsize=(10, 5))
plt.plot(month_names, monthly_delhi.values, marker='o',
         color='steelblue', linewidth=2.5, markersize=7)
plt.fill_between(month_names, monthly_delhi.values, alpha=0.15, color='steelblue')
plt.axhline(y=200, color='red', linestyle='--', linewidth=1, label='Poor threshold (200)')
plt.title('Monthly Average AQI — Delhi (2015–2020)', fontsize=14, fontweight='bold')
plt.xlabel('Month', fontsize=12)
plt.ylabel('Average AQI', fontsize=12)
plt.legend()
plt.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig('Figure_4_3_Monthly_Delhi.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved Figure 4.3")

In [ ]:
# Figure 4.4 — Year-wise AQI Trend (Top 5 Polluted Cities)
top5 = city_avg.sort_values(ascending=False).head(5).index.tolist()
yearly = df[df['City'].isin(top5)].groupby(['Year','City'])['AQI'].mean().reset_index()

plt.figure(figsize=(11, 6))
for city in top5:
    data = yearly[yearly['City'] == city]
    plt.plot(data['Year'], data['AQI'], marker='o', linewidth=2, label=city)

plt.title('Year-wise AQI Trend — Top 5 Most Polluted Cities', fontsize=14, fontweight='bold')
plt.xlabel('Year', fontsize=12)
plt.ylabel('Average AQI', fontsize=12)
plt.legend(title='City', fontsize=10)
plt.grid(alpha=0.4)
plt.tight_layout()
plt.savefig('Figure_4_4_Yearly_Trend.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved Figure 4.4")

In [ ]:
# Figure 4.5 — Pollutant Correlation Heatmap
pollutants = ['PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3', 'AQI']
corr = df[pollutants].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            linewidths=0.5, annot_kws={'size': 11},
            vmin=-1, vmax=1)
plt.title('Correlation Heatmap — Pollutants vs AQI', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('Figure_4_5_Correlation_Heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved Figure 4.5")

In [ ]:
# Figure 4.6 — AQI Box Plot by Season
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure 'Date' is datetime and extract 'Month' and 'Year' if not already done
if 'Date' in df.columns and not pd.api.types.is_datetime64_any_dtype(df['Date']):
    df['Date'] = pd.to_datetime(df['Date'])

if 'Month' not in df.columns and 'Date' in df.columns:
    df['Month'] = df['Date'].dt.month

if 'Year' not in df.columns and 'Date' in df.columns:
    df['Year'] = df['Date'].dt.year

# Re-create Season column if it doesn't exist
if 'Season' not in df.columns and 'Month' in df.columns:
    def get_season(month):
        if month in [10, 11, 12, 1, 2]:
            return 'Winter'
        elif month in [3, 4, 5]:
            return 'Summer'
        else:
            return 'Monsoon'
    df['Season'] = df['Month'].apply(get_season)

season_order = ['Winter', 'Summer', 'Monsoon']
palette = {'Winter': '#5b8dd9', 'Summer': '#e08a3c', 'Monsoon': '#5cad6e'}

plt.figure(figsize=(9, 6))
sns.boxplot(data=df, x='Season', y='AQI', order=season_order,
            palette=palette, width=0.5, fliersize=3)
plt.title('AQI Distribution by Season — All Cities', fontsize=14, fontweight='bold')
plt.xlabel('Season', fontsize=12)
plt.ylabel('AQI Value', fontsize=12)
plt.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig('Figure_4_6_Boxplot_Season.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved Figure 4.6")

In [ ]:
# Figure 4.7 — AQI Category Distribution Pie Chart
category_counts = df['AQI_Bucket'].value_counts()

colors_pie = ['#5cad6e','#a8d45e','#f0c040','#e08a3c','#e05c5c','#9b3535']
plt.figure(figsize=(9, 7))
wedges, texts, autotexts = plt.pie(
    category_counts.values,
    labels=category_counts.index,
    autopct='%1.1f%%',
    colors=colors_pie,
    startangle=140,
    pctdistance=0.82,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
for t in autotexts:
    t.set_fontsize(10)
plt.title('Distribution of AQI Categories — All Cities (2015–2020)',
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('Figure_4_7_AQI_Categories.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved Figure 4.7")

In [ ]:
# Figure 4.8 — Scatter Plot PM2.5 vs AQI
sample = df[['PM2.5','AQI']].dropna().sample(3000, random_state=42)

plt.figure(figsize=(9, 6))
plt.scatter(sample['PM2.5'], sample['AQI'], alpha=0.3, s=15, color='steelblue')

# Add trend line
z = np.polyfit(sample['PM2.5'], sample['AQI'], 1)
p = np.poly1d(z)
x_line = np.linspace(sample['PM2.5'].min(), sample['PM2.5'].max(), 200)
plt.plot(x_line, p(x_line), color='red', linewidth=2, label='Trend line')

# Pearson correlation
r, pval = stats.pearsonr(sample['PM2.5'], sample['AQI'])
plt.text(0.05, 0.92, f'r = {r:.2f},  p < 0.001',
         transform=plt.gca().transAxes, fontsize=11,
         bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='gray'))

plt.title('Scatter Plot: PM2.5 vs AQI', fontsize=14, fontweight='bold')
plt.xlabel('PM2.5 Concentration', fontsize=12)
plt.ylabel('AQI Value', fontsize=12)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('Figure_4_8_PM25_vs_AQI.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved Figure 4.8")

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

# Re-ensure 'Date' is datetime and extract 'Month' and 'Year' if not already done
if 'Date' in df.columns and not pd.api.types.is_datetime64_any_dtype(df['Date']):
    df['Date'] = pd.to_datetime(df['Date'])

if 'Month' not in df.columns and 'Date' in df.columns:
    df['Month'] = df['Date'].dt.month

if 'Year' not in df.columns and 'Date' in df.columns:
    df['Year'] = df['Date'].dt.year

# Re-create Season column if it doesn't exist
if 'Season' not in df.columns and 'Month' in df.columns:
    def get_season(month):
        if month in [10, 11, 12, 1, 2]:
            return 'Winter'
        elif month in [3, 4, 5]:
            return 'Summer'
        else:
            return 'Monsoon'
    df['Season'] = df['Month'].apply(get_season)

# Hypothesis 1 — Winter vs Summer AQI
winter_aqi = df[df['Season'] == 'Winter']['AQI'].dropna()
summer_aqi = df[df['Season'] == 'Summer']['AQI'].dropna()
t1, p1 = stats.ttest_ind(winter_aqi, summer_aqi)
print("=== Hypothesis 1: Winter vs Summer ===")
print(f"Winter mean AQI : {winter_aqi.mean():.2f}")
print(f"Summer mean AQI : {summer_aqi.mean():.2f}")
print(f"t = {t1:.2f},  p = {p1:.6f}")
print("Result: REJECT H0" if p1 < 0.05 else "Result: FAIL TO REJECT H0")

# Hypothesis 2 — PM2.5 correlation with AQI
pm_aqi = df[['PM2.5','AQI']].dropna()
r2, p2 = stats.pearsonr(pm_aqi['PM2.5'], pm_aqi['AQI'])
print("\n=== Hypothesis 2: PM2.5 correlation ===")
print(f"Pearson r = {r2:.4f},  p = {p2:.6f}")
print("Result: REJECT H0" if p2 < 0.05 else "Result: FAIL TO REJECT H0")

# Hypothesis 3 — North vs South cities
north = ['Delhi','Patna','Lucknow','Agra','Kanpur']
south = ['Thiruvananthapuram','Chennai','Hyderabad','Bengaluru','Vijayawada']
north_aqi = df[df['City'].isin(north)]['AQI'].dropna()
south_aqi = df[df['City'].isin(south)]['AQI'].dropna()
t3, p3 = stats.ttest_ind(north_aqi, south_aqi)
print("\n=== Hypothesis 3: North vs South ===")
print(f"North mean AQI : {north_aqi.mean():.2f}")
print(f"South mean AQI : {south_aqi.mean():.2f}")
print(f"t = {t3:.2f},  p = {p3:.6f}")
print("Result: REJECT H0" if p3 < 0.05 else "Result: FAIL TO REJECT H0")